In [ ]:
# run this notebook after 7_analyze_differences_twins_R
# use this notebook to analyze twin/duplicate spectra before and after removing giab difficult regions
# ensure you have hg38 reference fasta under hg38.fa.gz
# after this, run 9_find_differences_trios_python
# code to generate and plot spectra are from natanael spisak

In [ ]:
import pandas as pd
import os
from google.cloud import storage
import re
import numpy as np 
import pandas as pd
pd.options.mode.chained_assignment = None
import os
import pyfastx
from liftover import get_lifter as Converter
import matplotlib.pyplot as plt

In [ ]:
# from natanael spisak's github repo: https://github.com/n-t-n-el/clock-like

class Reference():
    """ This class uses genomic reference """
    
    def __init__(self,ref=19,dirname=None):
        assert (ref==19) or (ref==38)
        if dirname is not None: self.dirname = dirname
        else: dirname = 'data/'
        
        self.hg = Fasta(dirname + 'hg{}.fa'.format(ref))
        self.hgRefChr = {}
        for i in list(range(1,22+1)) + ['X', 'Y']:
            key = 'chr{}'.format(i)
            self.hgRefChr.update({str(i): key})
    
    def findRef(self,Chr,Pos):
        return self.hg[self.hgRefChr[Chr]][Pos-1].seq.upper()

    def findContext(self,Chr,Pos):
        return self.hg[self.hgRefChr[Chr]][Pos-2:Pos+1].seq.upper()

    def check(self,df,chr_col='Chr',pos_col='Pos',pos_shift=0,ref_col='Ref'):
        df['Chr'] = df[chr_col].astype(str).str.replace('chr','')
        df['Pos'] = df[pos_col]+pos_shift
        df['Ref-check'] = df.apply(lambda x: self.findRef(x['Chr'], x['Pos']), axis=1)
        self.errors = df['Ref-check']!=df[ref_col]
        return sum(df['Ref-check']!=df[ref_col])/len(df)

conv = Converter('hg38', 'hg19')
def convertChr(Chr,Pos):
    l = conv.convert_coordinate(Chr,Pos)
    if len(l)>0: return l[0][0].replace('chr','')
    else: return np.nan 
def convertPos(Chr,Pos):
    l = conv.convert_coordinate(Chr,Pos)
    if len(l)>0: return int(l[0][1])
    else: return np.nan 
    
    
class COSMIC():
    """ This class introduces cosmic order and 
        cosmic-like plots for signatures """
    
    def __init__(self,loc='data/COSMIC_v3.3.1_SBS_GRCh38.txt'):
        
        self.order = ['A[C>A]A', 'A[C>A]C', 'A[C>A]G', 'A[C>A]T',
                      'C[C>A]A', 'C[C>A]C', 'C[C>A]G', 'C[C>A]T',
                      'G[C>A]A', 'G[C>A]C', 'G[C>A]G', 'G[C>A]T',
                      'T[C>A]A', 'T[C>A]C', 'T[C>A]G', 'T[C>A]T',
                      'A[C>G]A', 'A[C>G]C', 'A[C>G]G', 'A[C>G]T',
                      'C[C>G]A', 'C[C>G]C', 'C[C>G]G', 'C[C>G]T',
                      'G[C>G]A', 'G[C>G]C', 'G[C>G]G', 'G[C>G]T',
                      'T[C>G]A', 'T[C>G]C', 'T[C>G]G', 'T[C>G]T',
                      'A[C>T]A', 'A[C>T]C', 'A[C>T]G', 'A[C>T]T',
                      'C[C>T]A', 'C[C>T]C', 'C[C>T]G', 'C[C>T]T',
                      'G[C>T]A', 'G[C>T]C', 'G[C>T]G', 'G[C>T]T',
                      'T[C>T]A', 'T[C>T]C', 'T[C>T]G', 'T[C>T]T',
                      'A[T>A]A', 'A[T>A]C', 'A[T>A]G', 'A[T>A]T',
                      'C[T>A]A', 'C[T>A]C', 'C[T>A]G', 'C[T>A]T',
                      'G[T>A]A', 'G[T>A]C', 'G[T>A]G', 'G[T>A]T',
                      'T[T>A]A', 'T[T>A]C', 'T[T>A]G', 'T[T>A]T',
                      'A[T>C]A', 'A[T>C]C', 'A[T>C]G', 'A[T>C]T',
                      'C[T>C]A', 'C[T>C]C', 'C[T>C]G', 'C[T>C]T',
                      'G[T>C]A', 'G[T>C]C', 'G[T>C]G', 'G[T>C]T',
                      'T[T>C]A', 'T[T>C]C', 'T[T>C]G', 'T[T>C]T',
                      'A[T>G]A', 'A[T>G]C', 'A[T>G]G', 'A[T>G]T',
                      'C[T>G]A', 'C[T>G]C', 'C[T>G]G', 'C[T>G]T',
                      'G[T>G]A', 'G[T>G]C', 'G[T>G]G', 'G[T>G]T',
                      'T[T>G]A', 'T[T>G]C', 'T[T>G]G', 'T[T>G]T']
        self.contexts = [t[0]+t[2]+t[6] for t in self.order]
        self.colors = [[3/256,189/256,239/256],
                       [1/256,1/256,1/256],
                       [228/256,41/256,38/256],
                       [203/256,202/256,202/256],
                       [162/256,207/256,99/256],
                       [236/256,199/256,197/256]]
        
        bad_order = pd.read_table(loc)
        good_order = np.array(self.order).argsort().argsort()
        self.cosmic = pd.DataFrame(bad_order.to_numpy()[good_order], columns=bad_order.columns)
        assert (self.cosmic.Type==self.order).all()
        A,C,G,T = 'A','C','G','T'
        self.RC = {A: T, C: G, G: C, T: A}
        self.Alt = {A: [C,G,T], C: [A,G,T], G: [A,C,T], T: [A,C,G]}

    
    def revType(self,typ):
        return self.RC[typ[6]]+typ[1]+self.RC[typ[2]]+typ[3]+self.RC[typ[4]]+typ[5]+self.RC[typ[0]]
        
    def prepareFold(self,types_column):
        generalTypes = set(types_column)
        self.foldDict = dict(zip(generalTypes, generalTypes))
        for typ in generalTypes:
            if typ[2] in ['A','G']:
                self.foldDict[typ] = self.revType(typ)
        
    def fold(self,types_column):
        """ Map 192 to 96 mutation types
            A[C>T]G is A[C>T]G (COSMIC red)
            but so is  C[G>A]T """ 
        self.prepareFold(types_column)
        return types_column.map(self.foldDict)
        
    def plot(self, proba, save=None, title=None, show_letters=False, figsize=(6,2), width=1):
        """COSMIC-like plot"""
        fig, ax = plt.subplots(figsize=figsize)

        delta = len(self.order)//len(self.colors)
        x = np.arange(delta)

        for i in range(6):
            ax.bar(x+i*delta, proba[i*delta:(i+1)*delta],
                   color=self.colors[i], width=width)

        ax.set_xlim(-width/2, len(self.order)-width/2)
        ax.set_ylim(0, 7)
        ax.set_xticks(range(len(self.order)))

        if show_letters:
            ax.set_xticklabels(self.contexts, rotation=90)
        else:
            ax.set_xticklabels([''] * len(self.contexts), rotation=90)

        ax.set_xlabel('Mutation types')
        ax.set_ylabel('Distribution')
        ax.legend([], title=title, frameon=False, loc=1)
        ax.set_ylim(0, 7)
        plt.tight_layout()
        if save:
            plt.savefig(save)
        plt.show()


In [ ]:
fa = pyfastx.Fasta('hg38.fa.gz')

In [ ]:
aou_difs = pd.read_table("./twins_dups_all_snps_after_standard_QC_oct16.txt", delimiter="\t")
# for after giab, aou_difs = pd.read_table("./aou_twin_differences_giab_removed.txt", delimiter="\t")
ukbb_difs = pd.read_table("./ukb/twins_diffs_full.txt", delimiter=" ")
# for after giab, ukbb_difs = pd.read_table("./ukb_twin_differences_giab_removed.txt", delimiter="\t")
ukbb_difs = ukbb_difs[(ukbb_difs['REF'].str.len() == 1) & (ukbb_difs['ALT'].str.len() == 1)]
ukbb_difs = ukbb_difs.reset_index(drop=True)

In [ ]:
cosmic = COSMIC()
aou_difs.loc[:, 'Context'] = ''
for i in range(aou_difs.shape[0]):
    chr_str = "chr" + str(aou_difs.loc[i,'chr'])
    Pos = aou_difs.loc[i,'pos']
    aou_difs.loc[i, 'Context'] = fa[chr_str][Pos-2:Pos+1].seq.upper()

In [ ]:
ukbb_difs.loc[:, 'Context'] = ''
for i in range(ukbb_difs.shape[0]):
    chr_str = "chr" + str(ukbb_difs.loc[i,'CHROM'])
    Pos = int(ukbb_difs.loc[i,'POS'])
    ukbb_difs.loc[i, 'Context'] = fa[chr_str][Pos-2:Pos+1].seq.upper()

In [ ]:
aou_difs[["ref", "alt"]] = aou_difs["alleles"].str.extract(r'\["([^"]+)"\s*,\s*"([^"]+)"\]')
aou_difs['GeneralType'] = aou_difs['Context'].str[0]+'['+aou_difs['ref']+'>'+aou_difs['alt']+']'+aou_difs['Context'].str[2]
aou_difs['Type'] = cosmic.fold(aou_difs['GeneralType'])
ukbb_difs['GeneralType'] = ukbb_difs['Context'].str[0]+'['+ukbb_difs['REF']+'>'+ukbb_difs['ALT']+']'+ukbb_difs['Context'].str[2]
ukbb_difs['Type'] = cosmic.fold(ukbb_difs['GeneralType'])

In [ ]:
cosmic = COSMIC()
cosmic_to_plot = pd.DataFrame(np.nan, index=range(len(cosmic.contexts)), columns=['context'])
cosmic_to_plot['context'] = cosmic.order
cosmic_to_plot['aou_percent'] = np.nan
cosmic_to_plot['ukbb_percent'] = np.nan
tot_aou = aou_difs.shape[0]
tot_ukbb = ukbb_difs.shape[0]
for i in range(len(cosmic.order)):
    count_aou = aou_difs[aou_difs['Type'] == cosmic.order[i]].shape[0]
    count_ukbb = ukbb_difs[ukbb_difs['Type'] == cosmic.order[i]].shape[0]
    cosmic_to_plot.loc[i,'aou_percent'] = count_aou/tot_aou*100
    cosmic_to_plot.loc[i,'ukbb_percent'] = count_ukbb/tot_ukbb*100
    
cosmic.plot(proba = cosmic_to_plot['aou_percent'], figsize=(12,4),
             width=0.7, show_letters = True, title="")

cosmic.plot(proba = cosmic_to_plot['ukbb_percent'], figsize=(12,4),
             width=0.7, show_letters = True, title="")

In [ ]:
#https://datastax.medium.com/how-to-implement-cosine-similarity-in-python-505e8ec1d823
A = cosmic_to_plot["aou_percent"]
B = cosmic_to_plot["ukbb_percent"]
dot_product = sum(a*b for a, b in zip(A, B))

# calculate the magnitude of each vector
magnitude_A = sum(a*a for a in A)**0.5
magnitude_B = sum(b*b for b in B)**0.5

# compute cosine similarity
cosine_similarity = dot_product / (magnitude_A * magnitude_B)
print(f"Cosine Similarity using standard Python: {cosine_similarity}")

In [ ]:
# from natanael spisak's github repo: https://github.com/n-t-n-el/clock-like

class EM():
    """ This class estimates global signature proportions
        (theta) using Expectation-Maximization """
    
    def __init__(self,df,cosmic,beta=0,
                 cutoff=None,
                 choice=None,
                 cosmicType='Type'):
        self.order = cosmic.order
        self.cosmicType = cosmicType
        if cutoff is not None: self.cosmic = cosmic.cosmic[cosmic.columns[:cutoff+1]]       
        elif choice is not None: self.cosmic = cosmic.cosmic[[self.cosmicType]+choice]       
        else: self.cosmic = cosmic.cosmic

        self.dim = len(self.cosmic.columns)-1
        self.theta = np.ones(self.dim)/self.dim
        self.x = df.groupby([self.cosmicType]).size().reindex(self.order, fill_value=0)
        # regularization: dirichlet prior with strength beta
        self.beta = beta 
        
    def expect(self,typ):
        """ Compute membership probabilities
            given theta """
        likelihood = self.cosmic.loc[self.cosmic.Type==typ].values[0,1:]
        membership = likelihood*self.theta
        return membership/membership.sum()
    
    def maximize(self):
        """ Update theta
            given membership probabilities """
        theta = np.zeros(self.dim,dtype=float)
        for typ in self.order:
            theta = theta + self.expect(typ)*self.x[typ]
        # regularization: dirichlet prior with strength beta
        theta -= self.beta
        # component annihilation 
        theta[theta<0] = 0
        self.theta = theta/theta.sum()

    def div(self,p1,p2):
        """ 1 - cos similarity """ 
        z = np.sqrt(np.sum(p1*p1)*np.sum(p2*p2))
        return 1 - np.sum(p1*p2)/z
    
    def infer(self,epsilon=1e-10):
        """ Iterative E-M """
        while True: 
            old_theta = self.theta
            self.maximize()
            if self.div(old_theta,self.theta) < epsilon: break
        
    def observed(self):
        """ Returns observed signature """
        ps = np.zeros(len(self.order))
        for i,typ in enumerate(self.order):
            ps[i] += self.x[typ]
        return ps/sum(ps)
    
    def reconstructed(self):
        """ Returns reconstructed signature """
        ps = (self.cosmic.values[:,1:].astype(float)*self.theta).sum(axis=1)
        return ps/sum(ps)
    
    def plot(self,
             figsize=(6,2),
             color=None,
             title=None,
             save=None,
             first=25):
        """ Plot global signature proportions """
        shift = 0.5
        width = 1
        fig = plt.figure(figsize=figsize)
        ax = fig.add_subplot(111)
        ax.bar(range(self.dim),self.theta[np.argsort(-self.theta)],
               color=color,width=width)
        ax.set_xlim(0-shift,first-shift)
        ax.set_ylim(0, 0.23)
        ax.set_xticks(range(first))
        ax.set_xticklabels(self.cosmic.columns[1:][np.argsort(-self.theta)][:first],
                           rotation=90)
        ax.set_ylabel('Distribution')
        ax.legend([],frameon=False,loc=1,title=title)
        plt.tight_layout()
        if save is not None: plt.savefig(save)
        plt.show()
        
    def signatureProba(self,signature,df):
        """ Estimate probability that mutation
            follow from given signature """
        assert signature in self.cosmic.columns
        index = np.where(self.cosmic.columns==signature)[0][0]-1
        ps = self.cosmic.values[:,1:]*self.theta # bayes rule
        probaDict = dict(zip(self.cosmic.Type,(ps.T/ps.sum(axis=1))[index]))
        return df[self.cosmicType].map(probaDict)
    
def EM_with_clock(df,
                  cosmic, 
                  intercept,
                  slope,
                  link='id',
                  choice=None,
                  cosmicType='Type',
                  ageLabel='Age',
                  beta=0):
    """ This function estimates clock-like signature proportions
        (theta) using Expectation-Maximization """

    order = cosmic.order
    if choice is not None:
        localcosmic = cosmic.cosmic[[cosmicType]+choice]
    else:
        localcosmic = cosmic.cosmic
    dim = len(localcosmic.columns) - 1
    theta = np.ones((2,dim),dtype=float) / dim
    
    data = pd.DataFrame(df.groupby([ageLabel, cosmicType]).size()).reset_index()
    t = data[ageLabel].values
    x = data[cosmicType].values
    count = data[0].values.astype(float)
    x_given_s = localcosmic.set_index('Type').loc[x].values.astype(float)
    if link=='id':
        intercept_given_t = (1/(slope/intercept*t+1)).astype(float)
        slope_given_t = 1 - intercept_given_t
    elif link=='log':
        intercept_given_t = np.exp(-slope*t).astype(float)
        slope_given_t = 1 - intercept_given_t
    
    """ Iterate until convergence """
    while True:
        old_theta = theta.copy()
        
        """ Compute membership probabilities given theta """
        intercept_membership = (x_given_s * theta[0]).T * intercept_given_t
        slope_membership = (x_given_s * theta[1]).T * slope_given_t
        Z = intercept_membership.sum(axis=0) + slope_membership.sum(axis=0)
        probabilities = np.array([intercept_membership,slope_membership])/Z

        """ Update theta given membership probabilities """
        theta = (probabilities * count).sum(axis=-1).T
        # regularization: dirichlet prior with strength beta
        theta -= beta
        # component annihilation 
        theta[theta<0] = 0
        theta = (theta/theta.sum(axis=0)).T
        
        if np.allclose(old_theta, theta, rtol=1e-4, atol=1e-4): break
    return theta

In [ ]:
em = EM(aou_difs, COSMIC(), beta = 0)
em.infer()

In [ ]:
em = EM(ukbb_difs, COSMIC(), beta = 0)
em.infer()